# LF1 — correctness độ chín (YOLOv8 cross-fitting)

Bỏ phiếu hữu dụng cho tác vụ `1_maturity_evaluation`. Xem thêm tại `docs/LF1_Methodology.md`.

Run All (Colab/Kaggle, cần GPU): cài ultralytics → train 5 fold cross-fitting → chấm correctness → ghi phiếu.
**Chống timeout:** train checkpoint từng fold (chạy lại → nạp fold đã xong), ghi phiếu incremental + resume.
Để công sức không mất khi hết session, đặt repo (Dataset + labels/ + runs) trên Google Drive.

In [25]:
%pip install -q ultralytics

Note: you may need to restart the kernel to use updated packages.


In [26]:
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass

## Cấu hình

In [27]:
import platform
import random
from pathlib import Path
import numpy as np
import pandas as pd

SEED       = 42
K          = 5
CONF_TAU   = 0.25
IOU_THR    = 0.5
YOLO_MODEL = 'yolov8n.pt'
EPOCHS     = 80
IMGSZ      = 640
BATCH      = 16
PATIENCE   = 15
KAGGLE_DATASET = 'duongthimyphuong/coconut'

TASK     = '1_maturity_evaluation'
CLASSES  = ['dry', 'green', 'tender']
STAGE    = {0: 'dry', 1: 'green', 2: 'tender'}
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

random.seed(SEED)
np.random.seed(SEED)

if Path('/kaggle').exists():
    RUNNER = 'Kaggle'
elif Path('/content').exists():
    RUNNER = 'Google Colab'
else:
    RUNNER = 'Local'

DEVICE = 'cpu'
TORCH_VERSION = 'not installed'
ULTRALYTICS_VERSION = 'not installed'
GPU_NAME = 'CPU'
try:
    import torch
    TORCH_VERSION = torch.__version__
    if torch.cuda.is_available():
        DEVICE = 0
        GPU_NAME = torch.cuda.get_device_name(0)
    elif torch.backends.mps.is_available():
        DEVICE = 'mps'
        GPU_NAME = 'mps'
except ImportError:
    pass
try:
    import ultralytics
    ULTRALYTICS_VERSION = ultralytics.__version__
except ImportError:
    pass

print('python:', platform.python_version(), '| numpy:', np.__version__, '| pandas:', pd.__version__)
print('torch:', TORCH_VERSION, '| ultralytics:', ULTRALYTICS_VERSION)
print('runner:', RUNNER, '| device:', DEVICE, '| gpu:', GPU_NAME)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/Users/peggy/Library/Application Support/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
python: 3.10.13 | numpy: 2.2.6 | pandas: 2.3.3
torch: 2.12.0 | ultralytics: 8.4.95
runner: Local | device: mps | gpu: mps


In [28]:
def veirf_dir_local():
    base = None
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent/'Dataset').is_dir():
            base = parent/'Dataset'
            break
    if base is None:
        raise SystemExit('Không thấy Dataset/ (local/Colab) — chạy notebook trong repo chứa Dataset/ (Colab: đặt repo trên Drive).')
    d = base/'coconut-veirf-v5'
    if not d.is_dir():
        raise SystemExit('Không thấy ' + str(d) + ' — kiểm tra lại đường dẫn.')
    return d

def veirf_dir_kaggle():
    d = Path('/kaggle/input/datasets')/KAGGLE_DATASET/'coconut-veirf-v5'/'coconut-veirf-v5'
    if not d.is_dir():
        raise SystemExit('Không thấy ' + str(d) + ' — kiểm tra lại dataset Kaggle đã add.')
    return d

def find_utils():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        cand = parent/'src'/'utils'/'lf_io.ipynb'
        if cand.exists():
            return cand
    for cand in Path('/kaggle/input').glob('**/src/utils/lf_io.ipynb'):
        return cand
    raise SystemExit('Không thấy src/utils/lf_io.ipynb — thêm repo vào runtime.')

if RUNNER == 'Kaggle':
    BASE = veirf_dir_kaggle()
    OUT_ROOT = Path('/kaggle/working')
else:
    BASE = veirf_dir_local()
    OUT_ROOT = BASE.parent.parent

VOTES_OUT = OUT_ROOT/'labels'/'votes'/'lf1_maturity.csv'
RUN_DIR   = OUT_ROOT/'labels'/'lf1_yolov8'/'runs'

# Module dùng chung (make_vote, append_lf_vote, done_image_ids, original_id, fold_of). %run vì .ipynb.
get_ipython().run_line_magic('run', str(find_utils()))

print('BASE:', BASE)
print('VOTES_OUT:', VOTES_OUT)
print('RUN_DIR:', RUN_DIR)

BASE: /Users/peggy/Documents/Projects/HK2/coconut-iqa/Dataset/coconut-veirf-v5
VOTES_OUT: /Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/votes/lf1_maturity.csv
RUN_DIR: /Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/lf1_yolov8/runs


/Users/peggy/.pyenv/versions/3.10.13/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


## 1. Đọc ground-truth (box + mức chín) + gán fold theo ảnh gốc

In [29]:
def read_boxes(txt):
    out = []
    if not txt.exists():
        return out
    for line in txt.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        box = (float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4]))
        out.append((cls, box))
    return out

def stage_set(boxes):
    stages = set()
    for cls, box in boxes:
        stages.add(STAGE[cls])
    return sorted(stages)

rows = []
for split in ('train', 'valid', 'test'):
    idir = BASE/split/'images'
    ldir = BASE/split/'labels'
    if not idir.exists():
        raise SystemExit(f'Không thấy {idir} — kiểm tra lại cấu trúc coconut-veirf-v5')
    for img in sorted(idir.iterdir()):
        if img.suffix.lower() not in IMG_EXTS:
            continue
        source = 'coconut-veirf-v5/' + split
        boxes = read_boxes(ldir/(img.stem + '.txt'))
        oid = original_id(img.stem, source)
        rows.append(dict(
            image_id    = img.stem,
            source      = source,
            path        = str(img.relative_to(BASE)),
            abspath     = str(img.resolve()),
            original_id = oid,
            fold        = fold_of(oid, K, SEED),
            gt_boxes    = boxes,
            gt_stages   = stage_set(boxes),
        ))
df = pd.DataFrame(rows)

spans = df.groupby('original_id').fold.nunique()
if not (spans == 1).all():
    raise SystemExit('RÒ RỈ: một ảnh gốc nằm ở nhiều fold')
box_stages = []
for r in rows:
    for cls, box in r['gt_boxes']:
        box_stages.append(STAGE[cls])
print('ảnh:', len(df), '| ảnh gốc:', df.original_id.nunique())
print('box theo mức:', pd.Series(box_stages).value_counts().to_dict())
for k in range(K):
    print('fold', k, '| ảnh', int((df.fold == k).sum()), '| ảnh gốc', df[df.fold == k].original_id.nunique())

ảnh: 948 | ảnh gốc: 399
box theo mức: {'tender': 685, 'green': 320, 'dry': 227}
fold 0 | ảnh 175 | ảnh gốc 75
fold 1 | ảnh 183 | ảnh gốc 76
fold 2 | ảnh 257 | ảnh gốc 109
fold 3 | ảnh 173 | ảnh gốc 67
fold 4 | ảnh 160 | ảnh gốc 72


## 2. Model out-of-fold + train cross-fitting YOLOv8

In [30]:
_FOLD_MODELS = {}

def register_fold_models(mapping):
    global _FOLD_MODELS
    _FOLD_MODELS = dict(mapping)

def predict_boxes(image_path, fold):
    # Trả list (cls, conf, (xc, yc, w, h) chuẩn hoá). Chưa đăng ký model fold -> [] (giữ chỗ).
    predictor = _FOLD_MODELS.get(fold)
    if predictor is None:
        return []
    return predictor(image_path)

def make_yolo_predictor(model, imgsz, device):
    def predict(image_path):
        results = model.predict(source=image_path, conf=0.01, imgsz=imgsz, device=device, verbose=False)
        boxes = []
        for out in results:
            for b in out.boxes:
                cls = int(b.cls[0])
                conf = float(b.conf[0])
                xywh = tuple(map(float, b.xywhn[0]))
                boxes.append((cls, conf, xywh))
        return boxes
    return predict

def train_oof_models(frame, run_dir):
    from ultralytics import YOLO
    run_dir = Path(run_dir)
    fold_models = {}
    for k in range(K):
        d = run_dir / ('fold' + str(k))
        weights = d / 'weights' / 'best.pt'
        if weights.exists():
            model = YOLO(str(weights))
            print('fold', k, '| nạp checkpoint (bỏ train)', flush=True)
        else:
            d.mkdir(parents=True, exist_ok=True)
            (d / 'train.txt').write_text('\n'.join(frame[frame.fold != k].abspath))
            (d / 'val.txt').write_text('\n'.join(frame[frame.fold == k].abspath))
            yaml_text = 'train: ' + str(d / 'train.txt') + '\nval: ' + str(d / 'val.txt') + '\nnc: ' + str(len(CLASSES)) + '\nnames: ' + str(CLASSES) + '\n'
            (d / 'data.yaml').write_text(yaml_text)
            model = YOLO(YOLO_MODEL)
            model.train(
                data=str(d / 'data.yaml'),
                epochs=EPOCHS,
                imgsz=IMGSZ,
                batch=BATCH,
                device=DEVICE,
                patience=PATIENCE,
                seed=SEED,
                project=str(run_dir),
                name='fold' + str(k),
                exist_ok=True,
                verbose=False,
            )
            print('fold', k, '| train xong', flush=True)
        fold_models[k] = make_yolo_predictor(model, IMGSZ, DEVICE)
    register_fold_models(fold_models)

In [31]:
train_oof_models(df, RUN_DIR)
print('model out-of-fold đã đăng ký:', len(_FOLD_MODELS), '/', K)

Ultralytics 8.4.95 🚀 Python-3.10.13 torch-2.12.0 MPS (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/lf1_yolov8/runs/fold0/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fold0, nbs=64, nms=False, opset=None, optimiz

## 3. Correctness (ghép box theo IoU) + cổng tin cậy + abstain

$$\lambda_1(x)=\begin{cases}\varnothing & G(x)=\emptyset \text{ hoặc } c(x)<\tau\\ 1 & \text{box tin cậy khớp GT theo IoU, đúng lớp}\\ 0 & \text{ngược lại}\end{cases}$$

In [32]:
def iou(a, b):
    ax1 = a[0] - a[2]/2
    ay1 = a[1] - a[3]/2
    ax2 = a[0] + a[2]/2
    ay2 = a[1] + a[3]/2
    bx1 = b[0] - b[2]/2
    by1 = b[1] - b[3]/2
    bx2 = b[0] + b[2]/2
    by2 = b[1] + b[3]/2
    iw = max(0.0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = a[2]*a[3] + b[2]*b[3] - inter
    if union <= 0:
        return 0.0
    return inter / union

def lf1_vote(gt_boxes, pred_boxes):
    # Trả (vote, pred_str, conf). vote None = abstain (dòng bị bỏ khi ghi).
    if not gt_boxes:
        return None, '', 0.0
    confident = []
    for p in pred_boxes:
        if p[1] >= CONF_TAU:
            confident.append(p)
    if not confident:
        return None, '', 0.0
    n_correct = 0
    n_wrong = 0
    conf_img = 0.0
    stages = set()
    for pcls, pconf, pbox in confident:
        if pconf > conf_img:
            conf_img = pconf
        stages.add(STAGE[pcls])
        ious = []
        for gcls, gbox in gt_boxes:
            ious.append(iou(pbox, gbox))
        j = int(np.argmax(ious))
        if ious[j] < IOU_THR:
            continue
        if pcls == gt_boxes[j][0]:
            n_correct = n_correct + 1
        else:
            n_wrong = n_wrong + 1
    pred_str = '|'.join(sorted(stages))
    if n_correct >= 1 and n_wrong == 0:
        return 1, pred_str, conf_img
    return 0, pred_str, conf_img

## 4. Ghi phiếu incremental + resume (`labels/votes/lf1_maturity.csv`)

In [33]:
done = done_image_ids(VOTES_OUT)
print('đã có', len(done), 'phiếu — resume, bỏ qua', flush=True)
n_written = 0
for r in df.itertuples():
    if r.image_id in done:
        continue
    pred_boxes = predict_boxes(r.abspath, r.fold)
    vote, pred_str, conf_img = lf1_vote(r.gt_boxes, pred_boxes)
    row = make_vote(
        lf='lf1_maturity',
        image_id=r.image_id,
        task=TASK,
        vote=vote,
        confidence=conf_img,
        source=r.source,
        path=r.path,
        pred=pred_str,
        fold=r.fold,
        original_id=r.original_id,
    )
    append_lf_vote(VOTES_OUT, row, extra_fields=['pred', 'fold', 'original_id'])
    if row is not None:
        n_written = n_written + 1
    print('xử lý', r.image_id, '| vote', vote, '| conf', round(conf_img, 3), flush=True)
print('ghi thêm', n_written, 'phiếu ->', VOTES_OUT, flush=True)

đã có 0 phiếu — resume, bỏ qua
xử lý 010_jpg.rf.0bceb2935b70017af4f5c8f966af09ab | vote 1 | conf 0.892
xử lý 010_jpg.rf.438e09acddb8e73d2bbce4e8dbe080a3 | vote 1 | conf 0.904
xử lý 010_jpg.rf.bc166106b70d5c7392f13edb897b33fe | vote 1 | conf 0.895
xử lý 015_jpg.rf.0696ae9f97d9a7e8f642fd504e1dfccc | vote 0 | conf 0.962
xử lý 015_jpg.rf.cfbcf01b383875e14bf9c7bb8921fe08 | vote 0 | conf 0.933
xử lý 015_jpg.rf.f0fe9af2487360f5ade10db38a21e87a | vote 0 | conf 0.964
xử lý 016_jpg.rf.1c75e3c14cf0b3190250b33769414d95 | vote 1 | conf 0.941
xử lý 016_jpg.rf.c307f930336c424f57b36d40e9eb758a | vote 1 | conf 0.944
xử lý 016_jpg.rf.e7e5eb3b47416c3c4ee2fb3aafe53aca | vote 1 | conf 0.93
xử lý 017_jpg.rf.56ee43ce0abdf695f5f4be4def2c97a1 | vote 1 | conf 0.935
xử lý 017_jpg.rf.744a099317cd7c34727ec5f74e028f51 | vote 1 | conf 0.933
xử lý 017_jpg.rf.d9a59ee6444d4cfbd3301079faa0dcc7 | vote 1 | conf 0.941
xử lý 01_jpg.rf.9a2fdd1fbadf3f0f94e9a35bf7f28f01 | vote 0 | conf 0.893
xử lý 01_jpg.rf.ece606812fdb5842eb9